## 1. Project Overview


CO3057 Computer Vision and Digital Image Preprocessing Project

## 2. Library Imports


In [5]:
import os
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import cv2
import cv2.aruco as aruco
import json
import glob

## 3. Global Constants and Configuration

In [6]:
DIRPATH = "./flyingarucov2"

ARUCO_DICT = aruco.getPredefinedDictionary(aruco.DICT_ARUCO_MIP_36h12)

## 4. Utility Functions

In [7]:
def load_data(img_id):
    img_path = os.path.join(DIRPATH, f"{img_id}.jpg")
    json_path = os.path.join(DIRPATH, f"{img_id}.json")

    img = cv2.imread(img_path)
    with open(json_path, 'r') as f:
        data = json.load(f)
    
    return img, data

In [11]:
def convert_to_yolo_file():
    label_dir = os.path.join(DIRPATH, 'labels')
    if os.path.exists(label_dir) and len(os.listdir(label_dir)) > 0:
        print(f'{label_dir} already exists.')
        return
        
    os.makedirs(label_dir, exist_ok=True)

    json_files = glob.glob(os.path.join(DIRPATH, '*.json'))

    for json_file in json_files:
        # 1. Lấy file ảnh
        file_id = os.path.basename(json_file).replace('.json', '')
        img_path = os.path.join(DIRPATH, f'{file_id}.jpg')

        # 2. Lấy shape để normalize
        img = cv2.imread(img_path)
        height, width, _ = img.shape

        # 3. Đọc file json
        with open(json_file, 'r') as f:
            data = json.load(f)

        lines = []
        for marker in data['markers']:
            corner = marker['corners']

            # Normalize data
            xs = [c[0] for c in corner]
            ys = [c[1] for c in corner]
                
            x_min, x_max = min(xs), max(xs)
            y_min, y_max = min(ys), max(ys)

            x_mid_norm = ((x_min + x_max) / 2) / height
            y_mid_norm = ((y_min + y_max) / 2) / width

            h_norm = (x_max - x_min) / height
            w_norm = (y_max - y_min) / width

            line = f'0 {x_mid_norm:.6f} {y_mid_norm:.6f} {h_norm:.6f} {w_norm:.6f}'
            lines.append(line)

        # 4. Ghi file
        txt_file = os.path.join(label_dir, f'{file_id}.txt')
        with open(txt_file, 'w') as f:
            f.write('\n'.join(lines))

convert_to_yolo_file()

./flyingarucov2\labels already exists.


In [12]:
import shutil
import random

def split_dataset(src_dir, train_ratio=.8):
    base_dir = "dataset"
    sub_dirs = ["images/train", "images/val", "labels/train", "labels/val"]

    for sub in sub_dirs:
        try:
            os.makedirs(os.path.join(base_dir, sub), exist_ok=True)
        except FileExistsError:
            print(f"Folder images and labels already exist")
            return
        
    all_ids = [f.replace('.jpg', '') for f in os.listdir(src_dir) if f.endswith('.jpg')]
    random.seed(42)
    random.shuffle(all_ids)

    split_index = int(len(all_ids) * train_ratio)
    train_ids = all_ids[:split_index]
    val_ids = all_ids[split_index:]

    def move_files(ids, split_name):
        label_src_dir = os.path.join(src_dir, "labels")
        for id in ids:
            # Di chuyển ảnh
            img_src = os.path.join(src_dir, f"{id}.jpg")
            img_dst = os.path.join(base_dir, f"images/{split_name}/{id}.jpg")
            if os.path.exists(img_src):
                shutil.copy(img_src, img_dst)

            txt_src = os.path.join(label_src_dir, f"{id}.txt")
            txt_dst = os.path.join(base_dir, f"labels/{split_name}/{id}.txt")
            if os.path.exists(txt_src):
                shutil.copy(txt_src, txt_dst)

    move_files(train_ids, "train")
    move_files(val_ids, "val")

split_dataset(DIRPATH)


## 5. Core

## 6. Evaluation